# Optimal control of spin experiments

Pulse sequences such as INEPT were designed by hand for ideal pulses.
Gradient ascent pulse engineering (GRAPE; Khaneja et al., J. Magn. Reson.
172, 296 (2005)) designs them numerically: the rf amplitudes of every
channel, or the field, are constant over many short steps, and all of
them are adjusted together to move the most polarisation where it is
wanted. QSO computes the exact gradient of the transfer with respect to
every step, with or without relaxation.

This notebook compares GRAPE with the limits that theory sets: the
largest transfer any unitary can achieve (Glaser et al., Science 280, 421
(1998)), and the shortest time a transfer between two coupled spins needs
(Khaneja, Brockett and Glaser, Phys. Rev. A 63, 032308 (2001)). Then it
designs a pulse that keeps working when the rf amplitude is off by 20 %,
and a field profile that carries parahydrogen order to ¹³C in two seconds.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import metatron_qso as q
from metatron_qso import spin

J = 140.0  # Hz, a one-bond ¹H–¹³C coupling


def pair(j=J):
    s = q.SpinSystem(9.4)
    s.add_spin("1H", 0.0)
    s.add_spin("13C", 0.0)
    s.couple(0, 1, j)
    return s


def best(system, control, seeds=3, scale=500.0, iterations=1000):
    runs = [system.optimize_control(control, seed=s, scale=scale, max_iterations=iterations) for s in range(seeds)]
    return max(runs, key=lambda r: r["transfer"])

## The unitary bound

At thermal equilibrium the proton is four times more polarised than
carbon. No unitary operation can put more on carbon than the sum of the
products of the ordered eigenvalues of the state and of the observable;
for a ¹H–¹³C pair that is the proton's polarisation. INEPT reaches it
with ideal pulses, and so does a GRAPE pulse of 1.2/J with rf limited to
5 kHz.

In [ ]:
s = pair()
rho = s.thermal_state()
sz = s.spin_operator(1, "z")
own = np.trace(rho @ sz).real
bound = spin.unitary_bound(rho, sz)
c = spin.Control.rf(["1H", "13C"], 1.2 / J, 40, rho, sz, limit=5000.0)
r = best(s, c)
print(f"carbon's own ⟨S_z⟩ {own:.3e}; bound {bound / own:.4f} times that; GRAPE {r['transfer'] / own:.4f}")

## How fast can polarisation move?

Between two spins coupled by `J`, transfer takes time. Khaneja, Brockett
and Glaser derived the largest transfer reachable in a time `T` with
arbitrarily strong pulses: antiphase coherence `2I_zS⁻` becomes `I⁻` at
most to `sin(πJT/2)` before `T = 1/J`. GRAPE with 20 kHz rf follows the
curve from below; at `JT = 0.5` it stays 0.6 % short, which finer steps
reduce.

In [ ]:
s = pair(100.0)
im = lambda i: s.spin_operator(i, "x") - 1j * s.spin_operator(i, "y")
antiphase = np.sqrt(2) * s.spin_operator(0, "z") @ im(1)
i_minus = im(0) / np.sqrt(2)
times = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2]
reached = []
for x in times:
    c = spin.Control.rf(["1H", "13C"], x / 100.0, 100, antiphase, i_minus, limit=20000.0)
    reached.append(best(s, c, seeds=2, scale=1000.0)["transfer"])
grid = np.linspace(0, 1.3, 200)
plt.plot(grid, np.sin(np.pi * np.minimum(grid, 1.0) / 2), label="time-optimal limit")
plt.plot(times, reached, "o", label="GRAPE")
plt.xlabel("JT")
plt.ylabel("transfer 2I_zS⁻ → I⁻")
plt.legend()
plt.show()

## Robust to rf errors

A coil's rf field is not the same across a sample. Here both channels
deliver between 80 % and 120 % of the nominal amplitude. Refocused INEPT
transfers everything only at 100 %; a GRAPE pulse optimised for five
amplitudes in that range transfers more than 99.99 % at all of them.

In [ ]:
s = pair()
iz, sz = s.spin_operator(0, "z"), s.spin_operator(1, "z")


def inept(scale):
    tau = 1 / (4 * J)
    p = lambda target, angle, phase=0.0: spin.pulse(target, angle * scale, phase)
    events = [p("1H", 90), spin.delay(tau), p("1H", 180), p("13C", 180), spin.delay(tau),
              p("1H", 90, 90), p("13C", 90), spin.delay(tau), p("1H", 180), p("13C", 180),
              spin.delay(tau), p("13C", 90, 270)]
    return np.trace(s.evolve(iz, events) @ sz).real


c = spin.Control.rf(["1H", "13C"], 1.5 / J, 60, iz, sz, limit=10000.0)
c.ensemble = [(1.0, [k, k], []) for k in (0.8, 0.9, 1.0, 1.1, 1.2)]
robust = best(s, c)
scales = np.linspace(0.7, 1.3, 61)
grape = []
for k in scales:
    c.ensemble = [(1.0, [k, k], [])]
    grape.append(s.control_transfer(c, robust["controls"])[0])
plt.plot(scales, [inept(k) for k in scales], label="refocused INEPT")
plt.plot(scales, grape, label="GRAPE, robust for 0.8–1.2")
plt.axvspan(0.8, 1.2, alpha=0.1)
plt.xlabel("rf amplitude / nominal")
plt.ylabel("⟨I_z⟩ → ⟨S_z⟩")
plt.legend()
plt.show()

## A field profile for parahydrogen

In the field-cycling experiment of the hyperpolarisation tutorial, two
protons from parahydrogen and a ¹³C were carried from zero field to
50 μT slowly enough to stay adiabatic: 50 s gave a ¹³C polarisation of
−0.813. A linear sweep of 2 s gives much less. GRAPE shapes the field of
a 2 s profile, in 40 constant steps, and recovers the adiabatic value.

In [ ]:
three = q.SpinSystem(0.0, frame="laboratory")
three.add_spin("1H", 2.0)
three.add_spin("1H", 4.0)
three.add_spin("13C", 170.0)
three.couple(0, 1, 7.2)
three.couple(0, 2, 5.8)
three.couple(1, 2, -1.4)
start = three.time_average(three.parahydrogen_state(0, 1))
minus_p = -2 * three.spin_operator(2, "z")
linear = three.evolve(start, [spin.sweep(5e-5, 2.0, 2000)])
print(f"linear sweep in 2 s: P(13C) = {-np.trace(linear @ minus_p).real:+.3f}")
f = spin.Control.field(0.0, 5e-5, 2.0, 40, start, minus_p)
profile = max((three.optimize_control(f, seed=k, max_iterations=1000) for k in range(3)), key=lambda r: r["transfer"])
print(f"optimised profile in 2 s: P(13C) = {-profile['transfer']:+.4f}")
t = np.arange(41) * 0.05
plt.step(t, np.append(profile["controls"], profile["controls"][-1]) * 1e6, where="post")
plt.xlabel("t (s)")
plt.ylabel("B (μT)")
plt.show()

The optimisation finds a local optimum from each random start; taking
the best of a few starts is usually enough for problems of this size.
With relaxation (`T₁`, `T₂` or Redfield theory) the same calls work in
Liouville space, at a higher cost per step.